In [0]:
------ TableList 
select C.CustomerName,C.CustomerCode,TL.SourceSchema,TL.SourceTablename, TL.ExtractionType, TL.WatermarkColumn, TL.LastExtractWatermark,CD.ConnectionName, CD.SecretKeyName, 'Bronze' as BronzeLayer
from TablesList as TL
left join ConnectionDetails as CD on TL.SourceConnectionID=CD.ConnectionID and TL.CustomerProductID=CD.CustomerProductID
left Join CustomerProduct as CP on CP.CustomerProductID=TL.CustomerProductID
left join customers as C on CP.CustomerID=C.CustomerID
where C.CustomerCode='BOSHOSP' and TL.isActive='true';

------ Secret Connection Details
select CD.ConnectionName, CD.SecretKeyName, C.CustomerName, C.CustomerCode, CD.IsActive from  connectiondetails as CD
left join clinicalforge.metadata.CustomerProduct as CP on CP.CustomerProductID=CD.CustomerProductID
left join clinicalforge.metadata.Customers as C on CP.CustomerID=C.CustomerID
where C.CustomerCode='BOSHOSP' and CD.isActive='true'

-----WH Tables List
select C.CustomerCode, C.CustomerName, CWHT.TargetSchema, CWHT.WarehouseTableName, CWHT.LoadMethod, CWHT.isActive, CWHT.WatermarkColumn from clinicalforge.metadata.customerwarehousetables as CWHT
left join clinicalforge.metadata.CustomerProduct as CP on CP.CustomerProductID=CWHT.CustomerProductID
left join clinicalforge.metadata.Customers as C on CP.CustomerID=C.CustomerID
where C.CustomerCode='BOSHOSP' and CWHT.isActive='true'

update customerwarehousetables
set LoadMethod='IncrementalWatermark'
where LoadMethod='FullRefresh'
---FullRefresh

--- BOSHOSP BostonHealthSystem
ALTER TABLE connectiondetails SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')

alter table connectiondetails rename column KeyVaultSecretName to SecretKeyName

update connectiondetails
set ConnectionName='boshosp-scope', SecretKeyName='boshosp-connection-json'
where ConnectionId=1



Alter table connectiondetails 
add columns (UserName varchar(100), Password varchar(100))


-------
select C.CustomerName,C.CustomerCode,TL.SourceSchema,TL.SourceTablename, TL.ExtractionType, TL.WatermarkColumn, TL.LastExtractWatermark,CD.ConnectionName, CD.SecretKeyName,TL.IsActive, 'Bronze' as BronzeLayer
from clinicalforge.metadata.TablesList as TL
left join clinicalforge.metadata.ConnectionDetails as CD on TL.SourceConnectionID=CD.ConnectionID and TL.CustomerProductID=CD.CustomerProductID
left Join clinicalforge.metadata.CustomerProduct as CP on CP.CustomerProductID=TL.CustomerProductID
left join clinicalforge.metadata.customers as C on CP.CustomerID=C.CustomerID
where C.CustomerCode='BOSHOSP' and TL.isActive='true';


---------
alter table clinicalforge.metadata.EventHubDetails rename column KeyVaultConnSecretStr to ConnectionStringEH

ALTER TABLE clinicalforge.metadata.EventHubDetails SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')

-------
select C.CustomerName, C.CustomerCode, EHD.NamespaceName, EHD.TopicName, EHD.ConnectionStringEH from clinicalforge.metadata.EventHubDetails as EHD
Left Join clinicalforge.metadata.CustomerProduct as CP on EHD.CustomerProductID=CP.CustomerProductID
Left Join clinicalforge.metadata.customers as C on CP.CustomerID=C.CustomerID
where C.CustomerCode='BOSHOSP' and C.isActive='true'

select * from clinicalforge.metadata.customerwarehousetables order by WarehouseTableName

-------
select C.CustomerCode, C.CustomerName, CWHT.TargetSchema, CWHT.WarehouseTableName, TL.SourceTableName, CWHT.LoadMethod, CWHT.isActive, CWHT.WatermarkColumn from clinicalforge.metadata.customerwarehousetables as CWHT
left join clinicalforge.metadata.TablesList as TL on CWHT.CustomerProductID=TL.CustomerProductID and CWHT.WarehouseTableName=TL.WarehouseTableName and CWHT.SourceTableID=TL.TableID
left join clinicalforge.metadata.CustomerProduct as CP on CP.CustomerProductID=CWHT.CustomerProductID
left join clinicalforge.metadata.Customers as C on CP.CustomerID=C.CustomerID
where C.CustomerCode='BOSHOSP' and CWHT.isActive='true'
order by  CWHT.WarehouseTableName


select * from clinicalforge.metadata.customerwarehousetables
select * from clinicalforge.metadata.EventHubDetails
select * from clinicalforge.metadata.tableslist where SourceSchema='Clinical'

-------
update clinicalforge.metadata.customerwarehousetables
set SourceTableName='Patients', SourceTableID=4
where WarehouseTableName='DimPatients';

update clinicalforge.metadata.customerwarehousetables
set SourceTableName='Appointments', SourceTableID=5
where WarehouseTableName='FactAppointments';

update clinicalforge.metadata.customerwarehousetables
set SourceTableName='Encounters', SourceTableID=6
where WarehouseTableName='FactEncounters';

update clinicalforge.metadata.customerwarehousetables
set SourceTableName='BedAssignments', SourceTableID=7
where WarehouseTableName='FactBedAssignments';

update clinicalforge.metadata.customerwarehousetables
set SourceTableName='MedicationOrders', SourceTableID=8
where WarehouseTableName='FactMedicationOrders';

update clinicalforge.metadata.customerwarehousetables
set SourceTableID=9, SourceTableName='LabOrders'
where WarehouseTableName='FactLabOrders';

update clinicalforge.metadata.customerwarehousetables
set SourceTableID=10, SourceTableName='Procedures'
where WarehouseTableName='FactProcedures';

update clinicalforge.metadata.customerwarehousetables
set SourceTableName='BillingInvoices', SourceTableID=11
where WarehouseTableName='FactBillingInvoices';

update clinicalforge.metadata.customerwarehousetables
set SourceTableName='Facilities', SourceTableID=1
where WarehouseTableName='DimFacilities';

update clinicalforge.metadata.customerwarehousetables
set SourceTableName='Beds', SourceTableID=2
where WarehouseTableName='DimBeds';

update clinicalforge.metadata.customerwarehousetables
set SourceTableName='Practitioners' , SourceTableID=3
where WarehouseTableName='DimPractitioners'

------
alter table clinicalforge.metadata.customerwarehousetables
add column  SourceTableName varchar(100), SourceTableID int

ALTER TABLE clinicalforge.metadata.customerwarehousetables 
ADD CONSTRAINT fk_source_tableid
FOREIGN KEY (SourceTableID) REFERENCES clinicalforge.metadata.tableslist(TableID);

------
select CWHT.WarehouseTableID, CWHT.SourceTableID,C.CustomerCode, C.CustomerName, CWHT.TargetSchema, CWHT.WarehouseTableName, TL.SourceTableName, CWHT.LoadMethod, CWHT.isActive, CWHT.WatermarkColumn, C.CustomerCode 
from clinicalforge.metadata.customerwarehousetables as CWHT
left join clinicalforge.metadata.TablesList as TL on CWHT.CustomerProductID=TL.CustomerProductID and CWHT.WarehouseTableName=TL.WarehouseTableName and CWHT.SourceTableID=TL.TableID
left join clinicalforge.metadata.CustomerProduct as CP on CP.CustomerProductID=CWHT.CustomerProductID
left join clinicalforge.metadata.Customers as C on CP.CustomerID=C.CustomerID
where C.CustomerCode='BOSHOSP' and CWHT.isActive='true' and CWHT.WatermarkColumn ='LastModifiedDateTime'
order by  CWHT.WarehouseTableID


--------
select COALESCE((SELECT MAX(CustomerID) FROM clinicalforge.metadata.Customers), 0) + 1;
select COALESCE((SELECT MAX(CustomerProductID) FROM clinicalforge.metadata.CustomerProduct), 0) + 1;
select COALESCE((SELECT MAX(EventHubID) FROM clinicalforge.metadata.EventHubDetails), 0) + 1;

select (SELECT MAX(ProductID) FROM clinicalforge.metadata.Products WHERE ProductCode = 'CLIN_FORGE' and ProductName='ClinicalIntelligenceForge');

select COALESCE((SELECT MAX(CustomerProductID) FROM clinicalforge.metadata.CustomerProduct), 0) + 1;


declare v_BaseTableID int;
set v_BaseTableID = COALESCE((SELECT MAX(TableID) FROM clinicalforge.metadata.TablesList), 0);

SELECT 
        v_BaseTableID + ROW_NUMBER() OVER(ORDER BY TableID) as TableID,
        2,
        2,
        SourceSchema,
        SourceTableName,
        WarehouseSchema,
        WarehouseTableName,
        ExtractionType,
        WatermarkColumn,
        CAST('1900-01-01 00:00:00' AS TIMESTAMP),
        DatabricksNotebookPath,
        true
    FROM clinicalforge.metadata.TablesList
    QUALIFY ROW_NUMBER() OVER (PARTITION BY SourceSchema, SourceTableName ORDER BY TableID DESC) = 1;



select * from customers